# nbskill behavior tests

This notebook is an executable contract for the public tools documented in the other notebooks. It builds temporary notebooks, edits them, runs them, reviews them, and converts a small Python file without touching the repository notebooks.

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb
from fastcore.nbio import write_nb as _write_nb
from fastcore.script import _in_call_parse

from nbskill.convert import py2nb
from nbskill.execute import exec_nb
from nbskill.foundation import demo_path, remove_demo_path
from nbskill.mcp import capture_call, create_mcp
from nbskill.read import nb_cell, nb_overview, show_doc
from nbskill.review import diff_nb
from nbskill.write import split_nb_chapter, update_cell, write_nb


def _find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    return start


project_root = _find_project_root()

## Build a notebook fixture

Every test below works against a temporary notebook. This mirrors the way agents should experiment: create a small fixture, prove the tool behavior, then apply the same tool to the real notebook.

In [ ]:
root = demo_path("test_nbskill_contract")
root.mkdir()
demo = root / "demo.ipynb"

_write_nb(new_nb([
    mk_cell("Notebook note.", cell_type="markdown"),
    mk_cell("import math", cell_type="code"),
    mk_cell(chr(10).join(["## Setup", "A markdown section for context."]), cell_type="markdown"),
    mk_cell(chr(10).join(["value = 3", "value"]), cell_type="code"),
    mk_cell(chr(10).join([
        "#| export",
        "def demo_fn(x):",
        "    \"\"\"Return the input.\"\"\"",
        "    return x",
        "",
        "class DemoBox:",
        "    \"\"\"Box a value.\"\"\"",
        "    def get(self):",
        "        \"\"\"Return the boxed value.\"\"\"",
        "        return 3",
    ]), cell_type="code"),
]), demo)
assert demo.exists()

## Reading and symbol documentation

The reader should expose useful notebook structure without raw JSON. `show_doc` should also find a symbol inside one of this repository's source notebooks and include the nearby rationale.

In [ ]:
overview = capture_call(nb_overview, nb_path=str(demo))
assert "Cell id=" in overview
assert "## Setup" in overview
assert "Notebook note." not in overview
assert "import math" in overview
assert "def demo_fn(x):" in overview
assert "Return the input." in overview
assert "class DemoBox:" in overview
assert "Box a value." in overview
assert "    def get(self):" in overview
assert "Return the boxed value." in overview
assert "1 |" not in overview

assert "Notebook note." in capture_call(nb_overview, nb_path=str(demo), include_docs=True)
out = StringIO()
token = _in_call_parse.set(False)
try:
    with redirect_stdout(out):
        silent_overview = nb_overview(str(demo), verbose=False)
finally:
    _in_call_parse.reset(token)
assert out.getvalue() == ""
assert "## Setup" in silent_overview

cell = capture_call(nb_cell, nb_path=str(demo), query='contains="value = 3"')
assert "## Setup" in cell
assert "1 | value = 3" in cell

doc = capture_call(show_doc, path=str(project_root / "nbs/01_read.ipynb"), symbol="nb_cell", context=1, source=False)
assert "Symbol nb_cell" in doc
assert "Definition" in doc

## Writing and targeted updates

The write path should append parsed Markdown and code blocks. The update path should use a stable cell id, then preserve the id while replacing the source.

In [ ]:
write_nb(
    str(demo),
    "%%markdown\n## Result\nThe next cell is edited by id.\n---\n%%code\nresult = value + 4\nresult",
)

result_cell = next(cell for cell in read_nb(demo).cells if "result = value + 4" in cell.source)
old_id = result_cell.id

update_cell(
    str(demo),
    "result = value + 5\nresult",
    cell_id=old_id,
)

updated = next(cell for cell in read_nb(demo).cells if cell.id == old_id)
assert "value + 5" in updated.source

## Execution and review output

Executing the fixture should store outputs in the notebook. Reviewing a repository notebook against itself should report no code-cell changes, which keeps review noise low.

In [ ]:
exec_nb(str(demo), timeout=5, show_output=False, allow_new=True)

executed = next(cell for cell in read_nb(demo).cells if cell.id == old_id)
texts = []
for output in executed.outputs:
    data = output.get("data", {})
    if "text/plain" in data:
        text = data["text/plain"]
        texts.append("".join(text) if isinstance(text, list) else str(text))
assert any("8" in text for text in texts)

diff_text = capture_call(diff_nb, path=str(project_root / "nbs/02_write.ipynb"), ref_a=None)
assert "No code cell changes" in diff_text

## Conversion and MCP construction

The converter should create a valid nbdev notebook from Python source, and the MCP factory should be testable without starting a server.

In [ ]:
sample_py = root / "sample_tool.py"
sample_py.write_text("def double(x):\n    return x * 2\n", encoding="utf-8")
converted = root / "sample_tool.ipynb"

py2nb(str(sample_py), dest=str(converted))
converted_nb = read_nb(converted)
assert converted.exists()
assert "#| default_exp sample_tool" in converted_nb.cells[0].source
assert any("def double" in cell.source for cell in converted_nb.cells)

mcp = create_mcp()
assert mcp is not None

remove_demo_path(root)

In [ ]:
def _write_split_contract_notebook(path):
    _write_nb(new_nb([
        mk_cell("#| default_exp split_source", cell_type="code"),
        mk_cell("import math", cell_type="code"),
        mk_cell(chr(10).join(["#| export", "def _helper(x):", "    return math.ceil(x)"]), cell_type="code"),
        mk_cell("## Feature", cell_type="markdown"),
        mk_cell(chr(10).join(["#| export", "def split_value(x):", "    return _helper(math.sqrt(x))"]), cell_type="code"),
        mk_cell("## Keep", cell_type="markdown"),
        mk_cell(chr(10).join(["#| export", "def keep_value(x):", "    return _helper(x)"]), cell_type="code"),
    ]), path)


In [ ]:
split_root = demo_path("test_nbskill_split")
try:
    split_root.mkdir()
    split_src = split_root / "split_source.ipynb"
    split_dest = split_root / "split_dest.ipynb"
    _write_split_contract_notebook(split_src)
    plan_text = capture_call(
        split_nb_chapter,
        path=str(split_src), chapter="Feature", dest=str(split_dest),
        default_exp="split_dest", dry_run=True,
    )
    assert "would split" in plan_text
    assert "_helper -> helper" in plan_text
    assert not split_dest.exists()
    split_nb_chapter(str(split_src), "Feature", str(split_dest), default_exp="split_dest", dry_run=False)
    src_text = "\n".join(cell.source for cell in read_nb(split_src).cells)
    dest_text = "\n".join(cell.source for cell in read_nb(split_dest).cells)
    assert "## Feature" not in src_text
    assert "## Keep" in src_text
    assert "def helper" in src_text
    assert "return helper(x)" in src_text
    assert "#| default_exp split_dest" in dest_text
    assert "import math" in dest_text
    assert "from nbskill.split_source import helper as _helper" in dest_text
    assert "def split_value" in dest_text
    assert "_helper(math.sqrt(x))" in dest_text
finally:
    remove_demo_path(split_root)


## MCP concurrency and wrapper failures

MCP tools should preserve notebook locking when clients issue parallel calls, and a failing wrapper should not leave the server unable to handle a later call for the same notebook.

In [ ]:
import asyncio
import threading
import time
import nbskill.mcp as _mcp_mod


In [ ]:
async def _exercise_mcp_parallel_reads(parallel_root):
    parallel_root.mkdir()
    same_path = parallel_root / "same.ipynb"
    other_path = parallel_root / "other.ipynb"
    third_path = parallel_root / "third.ipynb"
    for path in (same_path, other_path, third_path):
        _write_nb(new_nb([mk_cell("value = 1", cell_type="code")]), path)
    guard = threading.Lock()
    state = {"active_by_path": {}, "max_by_path": {}}
    old_nb_overview = _mcp_mod.nb_overview
    def fake_nb_overview(**kwargs):
        path = kwargs["nb_path"]
        with guard:
            state["active_by_path"][path] = state["active_by_path"].get(path, 0) + 1
            state["max_by_path"][path] = max(state["max_by_path"].get(path, 0), state["active_by_path"][path])
        time.sleep(0.1)
        print(f"read {Path(path).name}")
        with guard: state["active_by_path"][path] -= 1
    try:
        _mcp_mod.nb_overview = fake_nb_overview
        mcp = _mcp_mod.create_mcp()
        same_results = await asyncio.gather(mcp.call_tool("nb_overview", {"nb_path": str(same_path)}), mcp.call_tool("nb_overview", {"nb_path": str(same_path)}))
        assert state["max_by_path"][str(same_path)] == 1
        assert all("read same.ipynb" in result.structured_content["full_output"] for result in same_results)
        other_results = await asyncio.gather(mcp.call_tool("nb_overview", {"nb_path": str(other_path)}), mcp.call_tool("nb_overview", {"nb_path": str(third_path)}))
        assert {result.structured_content["full_output"] for result in other_results} == {"read other.ipynb", "read third.ipynb"}
    finally:
        _mcp_mod.nb_overview = old_nb_overview


In [ ]:
parallel_root = demo_path("test_mcp_parallel_calls")
try:
    await _exercise_mcp_parallel_reads(parallel_root)
finally:
    remove_demo_path(parallel_root)


In [ ]:
import nbskill.mcp as _mcp_mod

failure_root = demo_path("test_mcp_wrapper_failure")
try:
    failure_root.mkdir()
    failure_path = failure_root / "failure.ipynb"
    _write_nb(new_nb([mk_cell("value = 1", cell_type="code")]), failure_path)

    state = {"calls": 0}
    old_nb_overview = _mcp_mod.nb_overview

    def failing_then_recovered(**kwargs):
        state["calls"] += 1
        if state["calls"] == 1:
            raise RuntimeError("intentional wrapper failure")
        print("recovered")

    try:
        _mcp_mod.nb_overview = failing_then_recovered
        mcp = _mcp_mod.create_mcp()
        try:
            await mcp.call_tool("nb_overview", {"nb_path": str(failure_path)})
        except Exception as exc:
            assert "intentional wrapper failure" in str(exc)
        else:
            raise AssertionError("nb_overview failure should propagate to the MCP caller")

        recovered = await mcp.call_tool("nb_overview", {"nb_path": str(failure_path)})
        assert "recovered" in recovered.structured_content["full_output"]
        assert state["calls"] == 2
    finally:
        _mcp_mod.nb_overview = old_nb_overview
finally:
    remove_demo_path(failure_root)